# ReiLea RL Engine Training

Two-phase training pipeline for the ReiLea chess engine:

- **Phase 1 — Supervised Learning**: Behavior cloning from the PGN game database. Teaches ReiLea what moves look like in real games.
- **Phase 2 — RL Self-Play**: REINFORCE policy gradient, playing against Stockfish (Skill 10). Fine-tunes the policy beyond the supervised baseline.

Architecture: `ReiLeaNet` — 18-plane board input → 6 residual blocks (128 channels) → policy head (4096 classes) + value head (scalar).

## 1. Setup

In [1]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import chess
import chess.pgn
import chess.engine
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

from reilea import (
    ReiLeaNet, ReiLeaAgent,
    encode_board, move_to_index, index_to_move
)
from chess_game import GameDatabase, STOCKFISH_PATH

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)

DB_DIR = '../game_data'

Device: cpu


## 2. Data Loading

Parse every game in the PGN database into `(board_planes, move_index, value_target)` triples.

- `board_planes`: 18×8×8 float32 tensor
- `move_index`: integer in [0, 4095], encoding `from_sq * 64 + to_sq`
- `value_target`: +1.0 (current player won), 0.0 (draw), -1.0 (lost)

In [2]:
def result_to_value(result: str, turn: bool) -> float:
    """Game result string -> value from perspective of `turn` player."""
    if result == '1-0':
        return 1.0 if turn == chess.WHITE else -1.0
    elif result == '0-1':
        return -1.0 if turn == chess.WHITE else 1.0
    return 0.0


def extract_samples(game: chess.pgn.Game):
    result = game.headers.get('Result', '*')
    if result == '*':
        return []

    samples = []
    board = game.board()

    for move in game.mainline_moves():
        planes = encode_board(board)
        move_idx = move_to_index(move)
        value = result_to_value(result, board.turn)
        samples.append((planes, move_idx, value))
        board.push(move)

    return samples


def load_dataset(db_dir: str, max_games: int = None):
    db = GameDatabase(db_dir)
    states, actions, values = [], [], []
    for i, game in enumerate(db.load_all_games()):
        if max_games and i >= max_games:
            break
        for planes, move_idx, value in extract_samples(game):
            states.append(planes)
            actions.append(move_idx)
            values.append(value)
    print(f'Loaded {len(states):,} positions from {i+1} games')
    return (
        np.array(states, dtype=np.float32),
        np.array(actions, dtype=np.int64),
        np.array(values, dtype=np.float32),
    )


states, actions, values = load_dataset(DB_DIR)

Loaded 1,712 positions from 13 games


In [3]:
class ChessDataset(Dataset):
    def __init__(self, states, actions, values):
        self.states = torch.tensor(states)
        self.actions = torch.tensor(actions)
        self.values = torch.tensor(values).unsqueeze(1)

    def __len__(self):
        return len(self.states)

    def __getitem__(self, idx):
        return self.states[idx], self.actions[idx], self.values[idx]


BATCH_SIZE = 256
VAL_SPLIT = 0.05

n = len(states)
n_val = max(1, int(n * VAL_SPLIT))
perm = np.random.permutation(n)
train_idx, val_idx = perm[n_val:], perm[:n_val]

train_ds = ChessDataset(states[train_idx], actions[train_idx], values[train_idx])
val_ds   = ChessDataset(states[val_idx],   actions[val_idx],   values[val_idx])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train: {len(train_ds):,}  |  Val: {len(val_ds):,}')

Train: 1,627  |  Val: 85


## 3. Phase 1 — Supervised Training

Loss = `CrossEntropy(policy_logits, played_move)` + `MSE(value_pred, game_outcome)`

The policy head learns which moves are played in real games.  
The value head learns to predict game outcomes from board positions.

In [4]:
SL_EPOCHS    = 20
SL_LR        = 1e-3
VALUE_WEIGHT = 0.5

model = ReiLeaNet(num_res_blocks=6, channels=128).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=SL_LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SL_EPOCHS)
policy_loss_fn = nn.CrossEntropyLoss()
value_loss_fn  = nn.MSELoss()


def run_epoch(loader, train=True):
    model.train(train)
    total_policy, total_value, total, correct = 0.0, 0.0, 0, 0

    with torch.set_grad_enabled(train):
        for states_b, actions_b, values_b in loader:
            states_b  = states_b.to(DEVICE)
            actions_b = actions_b.to(DEVICE)
            values_b  = values_b.to(DEVICE)

            policy_logits, value_pred = model(states_b)

            pl = policy_loss_fn(policy_logits, actions_b)
            vl = value_loss_fn(value_pred, values_b)
            loss = pl + VALUE_WEIGHT * vl

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_policy += pl.item() * len(states_b)
            total_value  += vl.item() * len(states_b)
            total += len(states_b)
            correct += (policy_logits.argmax(1) == actions_b).sum().item()

    return total_policy / total, total_value / total, correct / total


best_val_loss = float('inf')
print(f'{'Epoch':>6} | {'P-Loss':>8} | {'V-Loss':>8} | {'Acc%':>6} | {'vP-Loss':>8} | {'vAcc%':>6}')
print('-' * 62)

for epoch in range(1, SL_EPOCHS + 1):
    t_pl, t_vl, t_acc = run_epoch(train_loader, train=True)
    v_pl, v_vl, v_acc = run_epoch(val_loader,   train=False)
    scheduler.step()

    val_loss = v_pl + VALUE_WEIGHT * v_vl
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODELS_DIR / 'reiLea_supervised.pt')

    print(f'{epoch:>6} | {t_pl:>8.4f} | {t_vl:>8.4f} | {t_acc*100:>5.1f}% | {v_pl:>8.4f} | {v_acc*100:>5.1f}%')

print(f'\nBest model saved → models/reiLea_supervised.pt')

 Epoch |   P-Loss |   V-Loss |   Acc% |  vP-Loss |  vAcc%
--------------------------------------------------------------
     1 |   8.3130 |   0.2932 |   0.1% |   8.2675 |   0.0%
     2 |   7.9417 |   0.2568 |   0.1% |   8.0786 |   0.0%
     3 |   7.4910 |   0.2257 |   0.9% |   7.9314 |   0.0%
     4 |   7.1143 |   0.1658 |   1.3% |   7.8255 |   0.0%
     5 |   6.8101 |   0.1018 |   1.5% |   7.7590 |   0.0%
     6 |   6.5369 |   0.0634 |   2.4% |   7.7231 |   0.0%
     7 |   6.2845 |   0.0380 |   4.7% |   7.6886 |   0.0%
     8 |   6.0501 |   0.0285 |   6.3% |   7.6950 |   0.0%
     9 |   5.8192 |   0.0236 |   9.5% |   7.6901 |   0.0%
    10 |   5.5939 |   0.0187 |  13.3% |   7.6785 |   0.0%
    11 |   5.3727 |   0.0165 |  19.5% |   7.6398 |   0.0%
    12 |   5.1800 |   0.0148 |  26.0% |   7.7547 |   0.0%
    13 |   5.0014 |   0.0136 |  33.2% |   7.7004 |   0.0%
    14 |   4.8587 |   0.0122 |  39.2% |   7.7888 |   0.0%
    15 |   4.7305 |   0.0114 |  43.5% |   7.7096 |   0.0%
    16 | 

## 4. Phase 2 — RL Self-Play (REINFORCE vs Stockfish)

ReiLea (with temperature sampling) plays vs Stockfish Skill 10.  
Policy gradient update per game:

```
advantage  = reward - value_pred.detach()
policy_loss = -mean(log_prob * advantage)
value_loss  =  mean((value_pred - reward)^2)
entropy_bonus = -mean(sum(p * log_p))
loss = policy_loss + 0.5 * value_loss - 0.01 * entropy_bonus
```

In [5]:
# Load best supervised model as starting point
model.load_state_dict(torch.load(MODELS_DIR / 'reiLea_supervised.pt', map_location=DEVICE))
model.train()

RL_EPOCHS      = 50
RL_LR          = 1e-4
TEMPERATURE    = 1.0
SF_SKILL       = 10
SF_TIME        = 0.1
ENTROPY_COEFF  = 0.01
VALUE_COEFF    = 0.5

rl_optimizer = torch.optim.Adam(model.parameters(), lr=RL_LR, weight_decay=1e-4)

db = GameDatabase(DB_DIR)

In [6]:
def play_episode_vs_sf(engine, reiLea_color: bool, temperature: float = 1.0):
    """Play one game. Returns list of (log_prob, value_pred, entropy) for ReiLea moves, plus final reward."""
    board = chess.Board()
    trajectory = []

    while not board.is_game_over():
        if board.turn == reiLea_color:
            planes = encode_board(board)
            x = torch.tensor(planes).unsqueeze(0).to(DEVICE)

            policy_logits, value_pred = model(x)

            legal_moves = list(board.legal_moves)
            legal_indices = torch.tensor([move_to_index(m) for m in legal_moves], device=DEVICE)
            legal_logits = policy_logits[0][legal_indices] / temperature

            log_probs = F.log_softmax(legal_logits, dim=0)
            probs = torch.exp(log_probs)
            entropy = -(probs * log_probs).sum()

            chosen = torch.multinomial(probs, 1).item()
            move = index_to_move(legal_indices[chosen].item(), board)

            trajectory.append((log_probs[chosen], value_pred[0, 0], entropy))
            board.push(move)
        else:
            result = engine.play(board, chess.engine.Limit(time=SF_TIME))
            board.push(result.move)

    outcome = board.result()
    reward = result_to_value(outcome, reiLea_color)
    return trajectory, reward, outcome


def rl_update(trajectory, reward):
    if not trajectory:
        return 0.0

    log_probs  = torch.stack([t[0] for t in trajectory])
    value_preds = torch.stack([t[1] for t in trajectory])
    entropies  = torch.stack([t[2] for t in trajectory])

    reward_t = torch.tensor(reward, dtype=torch.float32, device=DEVICE)
    advantage = (reward_t - value_preds.detach())

    policy_loss = -(log_probs * advantage).mean()
    value_loss  = F.mse_loss(value_preds, reward_t.expand_as(value_preds))
    entropy_loss = -entropies.mean()

    loss = policy_loss + VALUE_COEFF * value_loss + ENTROPY_COEFF * entropy_loss

    rl_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    rl_optimizer.step()

    return loss.item()

In [8]:
engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)
engine.configure({'Skill Level': SF_SKILL})

best_rl_winrate = -1.0
history = {'wins': 0, 'draws': 0, 'losses': 0, 'loss': []}

print(f'{'Ep':>4} | {'Result':>8} | {'Reward':>7} | {'Loss':>8} | {'W/D/L':>14}')
print('-' * 58)

try:
    for ep in range(1, RL_EPOCHS + 1):
        reiLea_color = chess.WHITE if ep % 2 == 1 else chess.BLACK

        trajectory, reward, outcome = play_episode_vs_sf(engine, reiLea_color, TEMPERATURE)
        loss = rl_update(trajectory, reward)
        history['loss'].append(loss)

        if reward > 0:
            history['wins'] += 1
        elif reward == 0:
            history['draws'] += 1
        else:
            history['losses'] += 1

        winrate = history['wins'] / ep
        color_str = 'W' if reiLea_color == chess.WHITE else 'B'
        wdl = f"{history['wins']}/{history['draws']}/{history['losses']}"
        print(f'{ep:>4} | {color_str} {outcome:>6} | {reward:>+6.1f} | {loss:>8.4f} | {wdl:>14}')

        if winrate > best_rl_winrate:
            best_rl_winrate = winrate
            torch.save(model.state_dict(), MODELS_DIR / 'reiLea_rl.pt')

finally:
    engine.quit()

print(f'\nBest model saved → models/reiLea_rl.pt  (win rate: {best_rl_winrate:.1%})')

  Ep |   Result |  Reward |     Loss |          W/D/L
----------------------------------------------------------
   1 | W    0-1 |   -1.0 |  -2.4292 |          0/0/1
   2 | B    1-0 |   -1.0 |  -1.2594 |          0/0/2
   3 | W    0-1 |   -1.0 |  -0.8734 |          0/0/3
   4 | B    1-0 |   -1.0 |  -0.3069 |          0/0/4
   5 | W    0-1 |   -1.0 |  -0.4107 |          0/0/5
   6 | B    1-0 |   -1.0 |  -0.1908 |          0/0/6
   7 | W    0-1 |   -1.0 |  -0.5550 |          0/0/7
   8 | B    1-0 |   -1.0 |  -0.1690 |          0/0/8
   9 | W    0-1 |   -1.0 |  -0.1478 |          0/0/9
  10 | B    1-0 |   -1.0 |  -0.1418 |         0/0/10
  11 | W    0-1 |   -1.0 |  -0.2823 |         0/0/11
  12 | B    1-0 |   -1.0 |  -0.1905 |         0/0/12
  13 | W    0-1 |   -1.0 |  -0.1458 |         0/0/13
  14 | B    1-0 |   -1.0 |  -0.0920 |         0/0/14
  15 | W    0-1 |   -1.0 |  -0.1116 |         0/0/15
  16 | B    1-0 |   -1.0 |  -0.1208 |         0/0/16
  17 | W    0-1 |   -1.0 |  -0.1642 |  

## 5. Evaluation — ReiLea vs Stockfish

Run N games against Stockfish at multiple skill levels. Reports W/D/L counts and win rate.

In [9]:
def evaluate(model_path: str, n_games: int = 20, sf_skill: int = 10):
    agent = ReiLeaAgent(model_path=model_path, device=DEVICE)

    engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)
    engine.configure({'Skill Level': sf_skill})

    wins, draws, losses = 0, 0, 0

    try:
        for i in range(n_games):
            board = chess.Board()
            reiLea_color = chess.WHITE if i % 2 == 0 else chess.BLACK

            while not board.is_game_over():
                if board.turn == reiLea_color:
                    move_uci = agent.predict_move(board)
                    board.push(chess.Move.from_uci(move_uci))
                else:
                    result = engine.play(board, chess.engine.Limit(time=0.1))
                    board.push(result.move)

            r = result_to_value(board.result(), reiLea_color)
            if r > 0:
                wins += 1
            elif r == 0:
                draws += 1
            else:
                losses += 1
    finally:
        engine.quit()

    total = wins + draws + losses
    print(f'vs Stockfish Skill {sf_skill} ({n_games} games):')
    print(f'  Wins: {wins}  Draws: {draws}  Losses: {losses}')
    print(f'  Win rate: {wins/total:.1%}  Score: {(wins + 0.5*draws)/total:.1%}')
    return wins, draws, losses


print('=== Supervised model ===')
evaluate(str(MODELS_DIR / 'reiLea_supervised.pt'), n_games=20, sf_skill=10)

print('\n=== RL model ===')
evaluate(str(MODELS_DIR / 'reiLea_rl.pt'), n_games=20, sf_skill=10)

=== Supervised model ===
vs Stockfish Skill 10 (20 games):
  Wins: 0  Draws: 0  Losses: 20
  Win rate: 0.0%  Score: 0.0%

=== RL model ===
vs Stockfish Skill 10 (20 games):
  Wins: 0  Draws: 0  Losses: 20
  Win rate: 0.0%  Score: 0.0%


(0, 0, 20)

## 6. Generate Training Games via API

Once the model is trained, use the FastAPI endpoints to generate ReiLea vs Stockfish games  
and store them in the database for future training rounds.

In [11]:
import requests

BASE = 'http://localhost:8000'

resp = requests.post(f'{BASE}/api/db/generate-reiLea-vs-sf', params={'n': 20, 'sf_skill': 10})
data = resp.json()
print(f"Generated {data['generated']} games")

results = {}
for g in data['games']:
    results[g['result']] = results.get(g['result'], 0) + 1
print('Results:', results)

Generated 20 games
Results: {'0-1': 20}
